In [ ]:
import jax.numpy as jnp
from scipy.stats import norm

# Bayesian Regression

Note: $$\beta = \frac{1}{\sigma^2}$$

In [33]:
Xtrain  = jnp.array([0,1,2,4,5]) #Define training data
ytrain = jnp.array([1,0.5,-0.1,-0.9,1.1]) #Define training targets

Xtest = jnp.array([3])

def design_matrix(x):
    return jnp.column_stack((jnp.ones(len(x)), x, x**2))

######## Parameters ########
sigma = 1/5
alpha = 1/2

D = 3


Phi = design_matrix(Xtrain)
Phi_star = design_matrix(Xtest)


## Likelihood - $p(y | w)$

### Equation

$$\prod^{N}_{n = 1} \mathcal{N}(y_{n}| f(\bf{x_{n}}| \bf{w}), \sigma^{2})$$
or
$$\mathcal{N}(\bf{y} | \bf{\Phi w}, \sigma^{2} \bf{I})$$

### Code

In [39]:
def likelihood(w):
    return jnp.prod(jnp.array([norm.pdf(x = ytrain, loc = design_matrix(jnp.array(Xtrain)) @ w, scale = sigma)]))

#Example
likelihood(jnp.array([1.2311242 , -1.2679785 ,  0.23120649]))

Array(0.00014322, dtype=float32)

## Log-likelihood - $\log{p(y | w)}$

### Equation

$$\sum^{N}_{n = 1} \log{\mathcal{N}(y_{n}| f(\bf{x_{n}}| \bf{w}), \sigma^{2})}$$

### Code

In [36]:
def log_likelihood(w):
    return jnp.sum(jnp.array([norm.logpdf(x = ytrain, loc = design_matrix(jnp.array(Xtrain)) @ w, scale = sigma)]))

#Example
log_likelihood(jnp.array([1.2311242 , -1.2679785 ,  0.23120649]))

Array(-8.851121, dtype=float32)

## Marginal Likelihood - $p(\bf{y})$

### Equation

$$p(\bf{y}) = \mathcal{N}(\bf{y} | 0, \sigma^2 \bf{I} + \alpha^{-1} \Phi \Phi^{T})$$

### Code

In [23]:
def marginal_likelihood(phi):
    N = phi.shape[0]
    cov = sigma**2 * jnp.eye(N) + alpha**-1 * (phi @ phi.T)  # N×N
    return jnp.zeros(N), cov

marginal_likelihood(phi = Phi)

(Array([0., 0., 0., 0., 0.], dtype=float32),
 Array([[   2.04,    2.  ,    2.  ,    2.  ,    2.  ],
        [   2.  ,    6.04,   14.  ,   42.  ,   62.  ],
        [   2.  ,   14.  ,   42.04,  146.  ,  222.  ],
        [   2.  ,   42.  ,  146.  ,  546.04,  842.  ],
        [   2.  ,   62.  ,  222.  ,  842.  , 1302.04]], dtype=float32))

## Maximum likelihood estimator - $w_{\text{MLE}}$

### Equation

$$\hat{w}_{\text{MLE}} = (\Phi^{T} \Phi)^{-1} \Phi^{T} \bm{y}$$

### Code

In [31]:
def w_MLE(phi):
    return jnp.linalg.solve(phi.T@phi, phi.T@ytrain).ravel()

w_MLE(phi=Phi)

Array([ 1.2698047 , -1.3099022 ,  0.23847395], dtype=float32)

## Maximum a posteriori - $w_{\text{MAP}}$

### Equation

$$w_{\text{MAP}} = \beta \bf{S} \Phi^{T} \bf{y}$$
where
$$\bf{S} = (\alpha \bf{I} + \beta \Phi^{T} \Phi)^{-1}$$

### Code

In [35]:
beta = 1 / sigma**2

def w_MAP(phi):
    return (beta*jnp.linalg.solve(alpha*jnp.identity(D) + beta*(phi.T@phi), phi.T)@ytrain).ravel()

w_MAP(phi = Phi)

Array([ 1.2311242 , -1.2679785 ,  0.23120649], dtype=float32)

## Posterior - $p(w | y)$

### Equation

$$\bf{m} = \beta \bf{S} \Phi^{T} \bf{y}$$
where
$$\bf{S} = (\alpha \bf{I} + \beta \Phi^{T} \Phi)^{-1}$$

### Code

In [13]:
def posterior(phi):
        inv_S0 = alpha*jnp.identity(D)
        A = inv_S0 + beta*(phi.T@phi)
        
        # compute mean and covariance 
        m = beta*jnp.linalg.solve(A,phi.T)@ytrain   # eq. (2) above
        S = jnp.linalg.inv(A) 
        
        return m,S

posterior(phi = Phi)

(Array([ 0.9470227 ,  0.2469165 , -0.6790359 ,  0.12686162], dtype=float32),
 Array([[ 0.03658523, -0.04402104,  0.01481672, -0.00149987],
        [-0.04402107,  0.13294227, -0.06430131,  0.00799767],
        [ 0.01481677, -0.06430135,  0.03584182, -0.00480549],
        [-0.00149988,  0.00799768, -0.0048055 ,  0.00066975]],      dtype=float32))

## Predictive posterior - $p(y^{*} | \bf{y}, \bf{x^{*}})$

### Equation

$$\mathcal{N}(y^{*} | \bf{m}^{T} \phi_{*}, \phi_{*}^{T}\bf{S}\phi_{*} + \sigma^{2})$$
where
$$\bf{m} = \beta \bf{S} \Phi^{T} \bf{y}$$
and
$$\bf{S} = (\alpha \bf{I} + \beta \Phi^{T} \Phi)^{-1}$$

### Code

In [17]:
def posterior_predctive_y(phi, phi_star):
    inv_S0 = alpha*jnp.identity(D)
    A = inv_S0 + beta*(phi.T@phi)
        
    # compute mean and covariance 
    m = beta*jnp.linalg.solve(A,phi.T)@ytrain   # eq. (2) above
    S = jnp.linalg.inv(A) 
        
    m_star = (phi_star@m).ravel()   
    S_star = jnp.diag(phi_star@S@phi_star.T) + sigma**2   
        
    return m_star, S_star


posterior_predctive_y(phi = Phi, phi_star=Phi_star)

(Array([-0.99828744], dtype=float32), Array([0.073962], dtype=float32))

## Predictive posterior - $p(f^{*} | \bf{y}, \bf{x^{*}})$

### Equations

$$\mathcal{N}(f^{*} | \bf{m}^{T} \phi_{*}, \phi_{*}^{T}\bf{S}\phi_{*})$$
where
$$\bf{m} = \beta \bf{S} \Phi^{T} \bf{y}$$
and
$$\bf{S} = (\alpha \bf{I} + \beta \Phi^{T} \Phi)^{-1}$$

In [18]:
def posterior_predctive_f(phi, phi_star):
    inv_S0 = alpha*jnp.identity(D)
    A = inv_S0 + beta*(phi.T@phi)
        
    # compute mean and covariance 
    m = beta*jnp.linalg.solve(A,phi.T)@ytrain   # eq. (2) above
    S = jnp.linalg.inv(A) 
        
    m_star = (phi_star@m).ravel()   
    S_star = jnp.diag(phi_star@S@phi_star.T)
        
    return m_star, S_star


posterior_predctive_f(phi = Phi, phi_star=Phi_star)

(Array([-0.99828744], dtype=float32), Array([0.033962], dtype=float32))